In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 3: physics-only diagnostic

Default parameters, no fit. Every training sol is run at its own Ls, tau and rover pixel and compared with REMS ground temperature at the bin centre. Residual = model - REMS.

In [ ]:
# --- config ---
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t

PARAMS = dict(g_diff=t.G_DIFF, f_ir=t.F_IR, ti_scale=t.TI_SCALE)   # defaults only in this notebook
SUSPECT_SOLS = (3763, 3778)    # DEM slope and elevation suspect; scored separately
LS_BIN_DEG = 30
OUT = paths.OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
print(PARAMS)

## Run

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "col", "pixel_id", "slope_deg",
                "aspect_deg", "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sols["lat_deg"] = t.lat_from_row(sols.row)
print(len(sols), "sols,", len(tbl), "scored rows")

Ts, n_spin = t.simulate_sols(sols.lat_deg, sols.ls_deg, sols.tau, sols.slope_deg, sols.aspect_deg,
                             sols.albedo_mean, sols.ti_themis_tiu, **PARAMS)
print("spin-up sols:", n_spin)

sol_index = pd.Series(np.arange(len(sols)), index=sols.sol)
tbl["model_k"] = Ts[sol_index[tbl.sol].to_numpy(), t.bin_centre_step(tbl.bin)]
tbl["resid_k"] = tbl.model_k - tbl.ground_temp_k
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
tbl["group"] = np.where(tbl.sol.between(*SUSPECT_SOLS), "suspect_terrain",
                        np.where(tbl.tau_is_placeholder, "placeholder_tau", "main"))
assert tbl.resid_k.notna().all()
print(tbl.group.value_counts().to_string())

## Results

In [ ]:
def summary(g):
    r = g.resid_k
    return pd.Series({"n": len(r), "bias_k": r.mean(), "rmse_k": np.sqrt((r ** 2).mean()),
                      "rems_mean_k": g.ground_temp_k.mean(), "model_mean_k": g.model_k.mean()})

by_group = tbl.groupby("group").apply(summary, include_groups=False)
main = tbl[tbl.group == "main"]
by_hour = main.groupby("ltst_hour").apply(summary, include_groups=False)
by_ls = main.groupby("ls_bin").apply(summary, include_groups=False)
by_block = pd.concat({"main": main.groupby("block").apply(summary, include_groups=False),
                      "all": tbl.groupby("block").apply(summary, include_groups=False)}, axis=1)
macro = by_block[("main", "rmse_k")].mean()

pd.options.display.float_format = "{:.2f}".format
print("By group (main = real tau, terrain trusted):"); print(by_group.to_string())
print(); print(f"main: pooled RMSE {np.sqrt((main.resid_k ** 2).mean()):.2f} K, bias {main.resid_k.mean():+.2f} K, "
      f"macro RMSE over blocks {macro:.2f} K")
print(); print("By block:"); print(by_block.to_string())
print(); print("By LTST hour (main):"); print(by_hour.to_string())
print(); print("By Ls bin (main):"); print(by_ls.to_string())

tbl.to_csv(OUT / "diagnostic_rows.csv", index=False)
by_group.to_csv(OUT / "diagnostic_by_group.csv")
by_hour.to_csv(OUT / "diagnostic_by_ltst_hour.csv")
by_ls.to_csv(OUT / "diagnostic_by_ls_bin.csv")
by_block.to_csv(OUT / "diagnostic_by_block.csv")

In [ ]:
# diurnal shape: hourly mean of REMS and model, main rows, by night/day
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
for name, g in tbl.groupby("group"):
    h = g.groupby("ltst_hour").resid_k
    ax.plot(h.mean().index, h.mean(), marker="o", ms=3, label=f"{name} bias")
ax.plot(by_hour.index, by_hour.rmse_k, "k--", label="main RMSE")
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="LTST hour", ylabel="K", title="Model - REMS by LTST hour")
ax.legend(fontsize=8)
ax = axes[1]
ax.bar(by_ls.index + LS_BIN_DEG / 2, by_ls.bias_k, width=LS_BIN_DEG * 0.8)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="Ls (deg)", ylabel="bias (K)", title="Model - REMS by 30-degree Ls bin (main)")
fig.tight_layout(); fig.savefig(OUT / "diagnostic_bias.png", dpi=120)

blocks = sorted(tbl.block.unique())
fig, axes = plt.subplots(3, 4, figsize=(14, 9), sharex=True, sharey=True)
for ax, b in zip(axes.flat, blocks):
    g = tbl[tbl.block == b].groupby("ltst_hour")[["ground_temp_k", "model_k"]].mean()
    ax.plot(g.index, g.ground_temp_k, "k.-", label="REMS")
    ax.plot(g.index, g.model_k, "r.-", label="model")
    s = tbl[tbl.block == b].sol
    ax.set_title(f"block {b}: sols {s.min()}-{s.max()}", fontsize=9)
for ax in axes.flat[len(blocks):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8)
fig.supxlabel("LTST hour"); fig.supylabel("ground temperature (K)")
fig.tight_layout(); fig.savefig(OUT / "diagnostic_diurnal_by_block.png", dpi=120)
print("saved to", OUT)